# Load Packages

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from os.path import join, exists
import matplotlib.pyplot as plt

# Import Functions
sys.path.append("../../")

from src.configs.octmnist_config import data_name, data_name_oods
from src.file_manager.filepath import FilePath
from src.file_manager.load_save_df import load_all_pred_dfs
from src.data_generator.oct_mnist import load_octmnist_data_dict
from src.data_generator.octdl import load_octdl_data_dict
from src.data_processing.ood_dataset_preprocessing import left_join_datasets
from src.evaluation.generate_expl import get_incorrect_test_predictions, get_explanation, show_explanation, get_explanation_other

from model_ue_dict import ModelClass_dict, ue_dict
from cur_seed import seed
seed = 2024

fp = FilePath(data_name=data_name, seed=seed)
fp_ood = FilePath(data_name=data_name_oods[0], seed=seed)
fp_ood2 = FilePath(data_name=data_name_oods[1], seed=seed)
fp_demo = "../../demonstration_images"

# Load Data

In [ ]:
# Load Data
data_dict = load_octmnist_data_dict(fp_preprocessed=fp.get_preprocessed_folder())
num_ori_test = len(data_dict["test_df"])
# Load OOD Data With Similar Classes
octdl_in_data_dict, _ = load_octdl_data_dict(fp_preprocessed=fp_ood2.get_preprocessed_folder())
# - Add Same Class Images into the Dataset
data_dict = left_join_datasets(data_dict, octdl_in_data_dict)

# Load Predictions

In [ ]:
pred_df = load_all_pred_dfs(fp, ModelClass_dict=ModelClass_dict)

# Get Incorrect Test Predictions

In [ ]:
incorrect_test_df = get_incorrect_test_predictions(pred_df, ue_dict)
incorrect_test_df

# Generate Explanation

In [ ]:
data_dict['classes']

In [ ]:
incorrect_test_df

In [ ]:
data_dict["test_df"][index]

In [ ]:
data_dict['classes']

In [ ]:
incorrect_test_df.loc[incorrect_test_df["index"]==index,"class"].values[0]

In [ ]:
index=270
print("Actual:", 
      data_dict['classes'][incorrect_test_df.loc[incorrect_test_df["index"]==index,"class"].values[0]])
print("Predict:", 
      data_dict['classes'][incorrect_test_df.loc[incorrect_test_df["index"]==index,"class_pred_label_resnet"].values[0]])
img_dict = get_explanation(
    data_dict, split="test_df", index=index, fp=fp, seed=seed, 
    gamma=1, thres=0.1, override=True)
show_explanation(img_dict)
plt.savefig(join(fp_demo, "egRUE.jpg"), bbox_inches="tight")

# Comparison of other XAI methods

## IG

In [ ]:
from src.evaluation.generate_expl import ig_expl_func
img_dict = get_explanation_other(
    expl_name="IG", expl_func=ig_expl_func,
    data_dict=data_dict, split="test_df", index=270, fp=fp, seed=seed, 
    gamma=1, thres=0.1, override=True
)
show_explanation(img_dict)
plt.savefig(join(fp_demo, "igRUE.jpg"), bbox_inches="tight")

## Saliency

In [ ]:
from src.evaluation.generate_expl import saliency_expl_func
img_dict = get_explanation_other(
    expl_name="Saliency", expl_func=saliency_expl_func,
    data_dict=data_dict, split="test_df", index=270, fp=fp, seed=seed, 
    gamma=1, thres=0.1, override=True
)
show_explanation(img_dict)
plt.savefig(join(fp_demo, "saliencyRUE.jpg"), bbox_inches="tight")

## Guided GradCam

In [ ]:
from src.evaluation.generate_expl import gc_expl_func
img_dict = get_explanation_other(
    expl_name="GuidedGradCam", expl_func=gc_expl_func,
    data_dict=data_dict, split="test_df", index=270, fp=fp, seed=seed, 
    gamma=1, thres=0.1, override=True
)
show_explanation(img_dict)
plt.savefig(join(fp_demo, "guidedgradcamRUE.jpg"), bbox_inches="tight")

## Guided Backprop

In [ ]:
from src.evaluation.generate_expl import gbp_expl_func
img_dict = get_explanation_other(
    expl_name="GuidedBackprop", expl_func=gbp_expl_func,
    data_dict=data_dict, split="test_df", index=270, fp=fp, seed=seed, 
    gamma=1, thres=0.1, override=True
)
show_explanation(img_dict)
plt.savefig(join(fp_demo, "guidedbackpropRUE.jpg"), bbox_inches="tight")